In [7]:
from dotenv import load_dotenv
import os
import sys
import tabulate

load_dotenv(dotenv_path='../.env')

JMAIL_API = os.getenv("JMAIL_API")

sys.path.append(os.path.abspath('..'))

from data_loader import fetch_and_clean_emails




In [8]:
df_mail = fetch_and_clean_emails(api_url=JMAIL_API)

print(df_mail[df_mail["subject"] == "Re: October 19, Heidelberg"].head(1).to_json())

{"id":{},"doc_id":{},"message_index":{},"sender":{},"subject":{},"to_recipients":{},"cc_recipients":{},"bcc_recipients":{},"sent_at":{},"content_markdown":{},"attachments":{},"account_email":{},"email_drop_id":{},"folder_path":{},"epstein_is_sender":{},"all_participants":{}}


 Analizzare shape, colonne, dtypes, null values e sample del dataset

In [9]:


## 1. Shape: Quante righe e colonne ci sono?
print("=== SHAPE DEL DATASET ===")
print(f"Righe: {df_mail.shape[0]}, Colonne: {df_mail.shape[1]}\n")

## 2 & 3. Colonne e Dtypes: Panoramica generale
print("=== INFO SULLE COLONNE E TIPI DI DATO ===")
# df.info() ti dà in un colpo solo l'elenco delle colonne, i dtype e i valori non nulli
df_mail.info()
print("\n")

## 4. Null Values: Quanti valori mancanti ci sono per ogni colonna?
print("=== VALORI NULLI PER COLONNA ===")
null_counts = df_mail.isnull().sum()
# Mostro solo le colonne che hanno effettivamente valori nulli (opzionale ma comodo)
print(null_counts[null_counts > 0])
print("\n")

## 5. Sample: Guardiamo 5 righe casuali per capire l'aspetto dei dati
print("=== SAMPLE DEL DATASET (5 righe) ===")
# Usa display() invece di print() in Jupyter per renderizzare una bella tabella HTML
display(df_mail.sample(5))

=== SHAPE DEL DATASET ===
Righe: 42, Colonne: 16

=== INFO SULLE COLONNE E TIPI DI DATO ===
<class 'pandas.DataFrame'>
Index: 42 entries, 2 to 99
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 42 non-null     str  
 1   doc_id             42 non-null     str  
 2   message_index      42 non-null     int64
 3   sender             42 non-null     str  
 4   subject            42 non-null     str  
 5   to_recipients      42 non-null     str  
 6   cc_recipients      42 non-null     str  
 7   bcc_recipients     33 non-null     str  
 8   sent_at            42 non-null     str  
 9   content_markdown   42 non-null     str  
 10  attachments        42 non-null     int64
 11  account_email      42 non-null     str  
 12  email_drop_id      42 non-null     str  
 13  folder_path        32 non-null     str  
 14  epstein_is_sender  42 non-null     bool 
 15  all_participants   42 non-null    

,id,doc_id,message_index,sender,subject,to_recipients,cc_recipients,bcc_recipients,sent_at,content_markdown,attachments,account_email,email_drop_id,folder_path,epstein_is_sender,all_participants
6,001612df62eb14194162f0a366793927,e7d35dec89fc8b6f9a988e766df31f0e,0,J. Epstein <jeeproject@yahoo.com>,Re: Barbro Ehnbom: SALSS 2006,"[""Cecilia Steen <cecilia.steen@gmail.com>""]",[],[],2006-07-10T18:13:50.000Z,i'll try\n\n--- Cecilia Steen <cecilia.steen@g...,0,jeeproject@yahoo.com,yahoo_2,Sent,True,"j. epstein <jeeproject@yahoo.com> [""cecilia s..."
78,0127c6f84d4e6913e9129cc7ec776678,418bf00e1d6d4ef5831f1e8eefa2572d,0,J. Epstein <jeeproject@yahoo.com>,Re: !!! Re: Love from Prague,"[""<petranemcova@mac.com>""]",[],[],2008-07-30T05:33:17.000Z,"skin care , colon cleansing, anti aging, purit...",0,jeeproject@yahoo.com,yahoo_2,Sent,True,"j. epstein <jeeproject@yahoo.com> [""<petranem..."
54,00da3a1923b543dcdd7a19065c1a7343,b7175b6ecfc3e2cb1367456588ee2441,0,Gmax <gmax1@ellmax.com>,(no subject),"[""<jeeproject@yahoo.com>""]",[],[],2008-02-28T02:12:35.000Z,Should I do these\n\nStock is currently at 88....,0,jeeproject@yahoo.com,yahoo_2,Inbox,False,"gmax <gmax1@ellmax.com> [""<jeeproject@yahoo.c..."
9,001df92b110e9da90631a66cf97a0a11,cffd46d5af8f9bc29fb07179ab1ca5c3,0,J. Epstein <jeeproject@yahoo.com>,Re:,"[""<gmax1@mindspring.com>""]",[],[],2007-02-19T20:50:39.000Z,both\n\n----- Original Message ----\nFrom: Gma...,0,jeeproject@yahoo.com,yahoo_2,Sent,True,"j. epstein <jeeproject@yahoo.com> [""<gmax1@mi..."
44,00a9d5a8e4e994a72482c0b394806059,2feb87a40bc457a47cb6148a86681589,0,Jay Lefkowitz <JLefkowitz@kirkland.com>,Re: responsibilities.,"[""Jeffrey Epstein <littlestjeff@yahoo.com>"",""K...","[""me <jeeproject@yahoo.com>"",""mike tein <tein@...",[],2008-06-02T00:14:03.000Z,We have no communication at all for now with A...,0,jeeproject@yahoo.com,yahoo_2,Inbox,False,"jay lefkowitz <jlefkowitz@kirkland.com> [""jef..."


In [10]:
import pandas as pd
import os

# 1. Estrazione DINAMICA delle informazioni dal DataFrame attuale
colonne = df_mail.columns.tolist()
tipi = df_mail.dtypes.astype(str).tolist()
valori_nulli = df_mail.isnull().sum().tolist()

# 2. Mappatura delle logiche di business (le tue decisioni)
# Usiamo un dizionario basato sul nome della colonna. 
# Se i dati cambiano, le tue decisioni rimangono valide.
business_rules = {
    "id": {"Descrizione": "Identificatore univoco del record", "La teniamo?": "Sì", "Motivo": "Chiave primaria per database e ID nodo in Neo4J."},
    "doc_id": {"Descrizione": "Identificatore del documento originale", "La teniamo?": "Forse", "Motivo": "Da valutare se serve per risalire alla fonte."},
    "message_index": {"Descrizione": "Indice sequenziale del messaggio", "La teniamo?": "No", "Motivo": "Ordinamento interno inutile per i nostri scopi."},
    "sender": {"Descrizione": "Indirizzo email del mittente", "La teniamo?": "Sì", "Motivo": "Nodi e archi in Neo4J."},
    "subject": {"Descrizione": "Oggetto dell'email", "La teniamo?": "Sì", "Motivo": "Testo chiave per clustering e NLP."},
    "to_recipients": {"Descrizione": "Indirizzi email dei destinatari primari", "La teniamo?": "Sì", "Motivo": "Archi verso i destinatari in Neo4J."},
    "cc_recipients": {"Descrizione": "Indirizzi email in copia (CC)", "La teniamo?": "Sì", "Motivo": "Connessioni secondarie nel grafo."},
    "bcc_recipients": {"Descrizione": "Indirizzi email in copia nascosta (BCC)", "La teniamo?": "No", "Motivo": "Valori nulli, complica l'analisi relazionale."},
    "sent_at": {"Descrizione": "Timestamp di invio", "La teniamo?": "Sì", "Motivo": "Dashboard (serie storiche)."},
    "content_markdown": {"Descrizione": "Corpo del messaggio", "La teniamo?": "Sì", "Motivo": "Testo primario per clustering."},
    "attachments": {"Descrizione": "Contatore allegati", "La teniamo?": "Forse", "Motivo": "Statistiche per la Dashboard."},
    "account_email": {"Descrizione": "Account proprietario mailbox", "La teniamo?": "Forse", "Motivo": "Ridondante se account singolo."},
    "email_drop_id": {"Descrizione": "Batch di estrazione", "La teniamo?": "No", "Motivo": "Metadato tecnico inutile."},
    "folder_path": {"Descrizione": "Percorso cartella", "La teniamo?": "No", "Motivo": "Poco valore semantico."},
    "epstein_is_sender": {"Descrizione": "Flag mittente Epstein", "La teniamo?": "Sì", "Motivo": "Filtro rapido."},
    "all_participants": {"Descrizione": "Tutti i partecipanti", "La teniamo?": "Forse", "Motivo": "Possibile scorciatoia, ma ridondante."}
}

# 3. Creiamo le liste incrociando le colonne estratte dinamicamente con le tue regole
# Il '.get()' con fallback a "DA DEFINIRE" ti protegge se in futuro appare una colonna nuova non prevista
descrizioni = [business_rules.get(col, {}).get("Descrizione", "DA DEFINIRE") for col in colonne]
teniamo = [business_rules.get(col, {}).get("La teniamo?", "DA DEFINIRE") for col in colonne]
motivi = [business_rules.get(col, {}).get("Motivo", "DA DEFINIRE") for col in colonne]

# 4. Creiamo il DataFrame finale
df_notes = pd.DataFrame({
    "Colonna": colonne,
    "Descrizione": descrizioni,
    "Tipo": tipi,
    "Valori Nulli": valori_nulli,
    "La teniamo?": teniamo,
    "Motivo": motivi
})

# Visualizzazione
display(df_notes)

print("Tabella generata dinamicamente e salvata!")

,Colonna,Descrizione,Tipo,Valori Nulli,La teniamo?,Motivo
0,id,Identificatore univoco del record,str,0,Sì,Chiave primaria per database e ID nodo in Neo4J.
1,doc_id,Identificatore del documento originale,str,0,Forse,Da valutare se serve per risalire alla fonte.
2,message_index,Indice sequenziale del messaggio,int64,0,No,Ordinamento interno inutile per i nostri scopi.
3,sender,Indirizzo email del mittente,str,0,Sì,Nodi e archi in Neo4J.
4,subject,Oggetto dell'email,str,0,Sì,Testo chiave per clustering e NLP.
5,to_recipients,Indirizzi email dei destinatari primari,str,0,Sì,Archi verso i destinatari in Neo4J.
6,cc_recipients,Indirizzi email in copia (CC),str,0,Sì,Connessioni secondarie nel grafo.
7,bcc_recipients,Indirizzi email in copia nascosta (BCC),str,9,No,"Valori nulli, complica l'analisi relazionale."
8,sent_at,Timestamp di invio,str,0,Sì,Dashboard (serie storiche).
9,content_markdown,Corpo del messaggio,str,0,Sì,Testo primario per clustering.


Tabella generata dinamicamente e salvata!


In [11]:
import os

# --- TASK 1: Salvare il sample locale in Parquet ---
# Crea la cartella data/raw se non esiste
os.makedirs('/home/filippo/Scrivania/pizza-cluster/data/raw', exist_ok=True)

percorso_parquet = '/home/filippo/Scrivania/pizza-cluster/data/raw/emails_sample.parquet'
# Salviamo il DataFrame. index=False evita di salvare l'indice di Pandas come colonna extra
df_mail.to_parquet(percorso_parquet, index=False)
print(f"✅ Sample Parquet salvato con successo in: {percorso_parquet}")


# --- TASK 2: Creare il report Markdown ---
# Crea la cartella reports se non esiste
os.makedirs('/home/filippo/Scrivania/pizza-cluster/reports', exist_ok=True)

percorso_md = '/home/filippo/Scrivania/pizza-cluster/reports/dataset_notes.md'
# Apre il file in modalità scrittura ('w' crea il file automaticamente)
with open(percorso_md, 'w') as f:
    f.write("# Dataset Notes - JMAIL\n\n")
    f.write("## Analisi delle Colonne\n\n")
    # Inseriamo la tabella generata allo step precedente
    f.write(df_notes.to_markdown(index=False))

print(f"✅ Report Markdown creato e salvato con successo in: {percorso_md}")

✅ Sample Parquet salvato con successo in: /home/filippo/Scrivania/pizza-cluster/data/raw/emails_sample.parquet
✅ Report Markdown creato e salvato con successo in: /home/filippo/Scrivania/pizza-cluster/reports/dataset_notes.md
